In this notebook we showcase how to make use of the different elements of point_milling_frontend.py

In [1]:
import point_milling_frontend as pmf
import numpy as np
import plotly.graph_objects as go

First, we need a to define the geometry we are going to work with. Such geometry is defined by a 4x4 matrix that will work as the control points of the bicubic bezier patch. Then, we will define the matrices for the G_function ($G$), the rotation function ($\phi$) and the tilt function ($\theta$)

In [2]:
# initialize a seed for reproducibility
np.random.seed(2)

# Control points of the bicubic bezier patch representing the machining surface
Q = 0.1*np.array([[0,2,-2,0], [1,0, 0, 1], [-1,1, 0.5, 1], [0,0,0,0]])
mat_Q = Q+Q.T
# Rotation function matrix
# In this case, we want the tool to go 'against the grain' of the surface, hence the rotation value is set to constant 0.5*Pi
mat_phi = 0.5*np.pi*np.ones((4,4))
# Tilt function matrix
# Let's make it random but relatively flat so that no crazy behaviors happen
# The values are in the range [0, 0.5*Pi], so we want the tilt values to be around 0.25*Pi
mat_theta = 0.25*np.pi + 0.1*(np.random.rand(4,4))
# The G_function is going to produce straight lines as levelsets
mat_G = np.array(
    [[0, 0, 0, 0],
     [1, 1, 1, 1],
     [2, 2, 2, 2],
     [3, 3, 3, 3]]
)

In [3]:
real_surface_side_length = 100 # The side length of the surface is 100mm

# The tool radius is going to be 2cm
real_tool_radius = 20 # in mm
real_machining_tolerance =  0.05# 50 microns in mm
tool_radius = real_tool_radius / real_surface_side_length
machining_tolerance = real_machining_tolerance / real_surface_side_length
# offset distance m
m = 0
number_of_paths = 6
h = 0.01 # 1% of the width of the surface
surfaces_resolution = 100 # 100x100 points per surface
# Create the MachiningParameters object
machining_object = pmf.MachiningParameters(
                        R = tool_radius,
                        m = m, mat_Q = mat_Q,
                        matrices = [mat_G, mat_phi, mat_theta],
                        number_of_paths = number_of_paths,
                        h = h,
                        surfaces_resolution = surfaces_resolution,
                        machining_tolerance = machining_tolerance,
                        shank_length = 1+np.sqrt(5),
                        n_shanks = 10
                    )

In [ ]:
# show the geometry
fig = go.Figure() # initialize the figure

showheatmap = True

# Add the surface mesh
fig.add_mesh3d(
    x = machining_object.surface_vertices[:, 0],
    y = machining_object.surface_vertices[:, 1],
    z = machining_object.surface_vertices[:, 2],
    i = machining_object.surface_triangles[:, 0],
    j = machining_object.surface_triangles[:, 1],
    k = machining_object.surface_triangles[:, 2],
    showlegend=True,
    name='Surface'
)

# Add the contact curves
X = []
Y = []
Z = []
for env in machining_object.envelopes:
    x,y,z = env.curve(np.linspace(0,1,100))
    X.append(x)
    Y.append(y)
    Z.append(z)
X, Y, Z = np.concatenate(X), np.concatenate(Y), np.concatenate(Z)
fig.add_scatter3d(
    x = X,
    y = Y,
    z = Z,
    mode='markers',
    name='Contact Curves',
    marker=dict(
        size=2,
        color='black',
    ),
    showlegend=True
)

# Add the envelopes
concatenated_envelopes = machining_object.concatenated_envelopes_fun(method='from0to1')
envelopes_vertices = np.asarray(concatenated_envelopes.vertices)
envelopes_triangles = np.asarray(concatenated_envelopes.triangles)
fig.add_mesh3d(
    x = envelopes_vertices[:, 0],
    y = envelopes_vertices[:, 1],
    z = envelopes_vertices[:, 2],
    i = envelopes_triangles[:, 0],
    j = envelopes_triangles[:, 1],
    k = envelopes_triangles[:, 2],
    name='Envelopes',
    opacity=0.5,
    color='red',
    showscale=False,
    showlegend=True
)
# compute the error heatmap and show it only if showheatmap is True
if showheatmap == True:
    errores = real_surface_side_length*machining_object.error_measure_per_vertex()
    colorscale = 'turbo_r'

    cmin, cmax =  real_machining_tolerance*np.array([-1,1,])
    distances_adjusted_to_tolerances = errores.copy()
    distances_adjusted_to_tolerances[distances_adjusted_to_tolerances > cmax] = cmax # set undercut
    distances_adjusted_to_tolerances[distances_adjusted_to_tolerances < cmin] = cmin # set overcut

    fig.add_mesh3d(
        x = machining_object.surface_vertices[:, 0],
        y = machining_object.surface_vertices[:, 1],
        z = machining_object.surface_vertices[:, 2],
        colorscale = colorscale,
        cmin=cmin,
        cmax=cmax,
        intensity=distances_adjusted_to_tolerances,

        i = machining_object.surface_triangles[:, 0],
        j = machining_object.surface_triangles[:, 1],
        k = machining_object.surface_triangles[:, 2],
        showscale=True,
        showlegend=True,
        name = 'Machining Error Heatmap'
    )
fig.update_layout(
    showlegend = True,
    scene=dict(
        aspectmode='data'),
        width = 900,
        height = 750
    )

If we now wish to change the matrices, we do not need to create a new object, just update the one we have

In [ ]:
mat_G = np.array(
    [[0, 0, 0, 0],
     [0, 1, 2, 3],
     [0, 2, 2, 6],
     [0, 3, 6, 9]])
mat_phi = 0.5*np.pi*np.ones((4,4)) + 1*np.random.rand(4,4) # small random perturbation, recall rotation ranges from 0 to Pi
mat_theta = 0.25*np.pi + 0.5*(np.random.rand(4,4))

matrices = [mat_G, mat_phi, mat_theta]
machining_object.matrices = matrices

And now we can just repeat the same plot

In [ ]:
# show the geometry
fig = go.Figure() # initialize the figure

showheatmap = True

# Add the surface mesh
fig.add_mesh3d(
    x = machining_object.surface_vertices[:, 0],
    y = machining_object.surface_vertices[:, 1],
    z = machining_object.surface_vertices[:, 2],
    i = machining_object.surface_triangles[:, 0],
    j = machining_object.surface_triangles[:, 1],
    k = machining_object.surface_triangles[:, 2],
    showlegend=True,
    name='Surface'
)

# Add the contact curves
X = []
Y = []
Z = []
for env in machining_object.envelopes:
    x,y,z = env.curve(np.linspace(0,1,100))
    X.append(x)
    Y.append(y)
    Z.append(z)
X, Y, Z = np.concatenate(X), np.concatenate(Y), np.concatenate(Z)
fig.add_scatter3d(
    x = X,
    y = Y,
    z = Z,
    mode='markers',
    name='Contact Curves',
    marker=dict(
        size=2,
        color='black',
    ),
    showlegend=True
)

# Add the envelopes
concatenated_envelopes = machining_object.concatenated_envelopes_fun(method='from0to1')
envelopes_vertices = np.asarray(concatenated_envelopes.vertices)
envelopes_triangles = np.asarray(concatenated_envelopes.triangles)
fig.add_mesh3d(
    x = envelopes_vertices[:, 0],
    y = envelopes_vertices[:, 1],
    z = envelopes_vertices[:, 2],
    i = envelopes_triangles[:, 0],
    j = envelopes_triangles[:, 1],
    k = envelopes_triangles[:, 2],
    name='Envelopes',
    opacity=0.5,
    color='red',
    showscale=False,
    showlegend=True
)
# compute the error heatmap and show it only if showheatmap is True
if showheatmap == True:
    errores = real_surface_side_length*machining_object.error_measure_per_vertex()
    colorscale = 'turbo_r'

    cmin, cmax =  real_machining_tolerance*np.array([-1,1,])
    distances_adjusted_to_tolerances = errores.copy()
    distances_adjusted_to_tolerances[distances_adjusted_to_tolerances > cmax] = cmax # set undercut
    distances_adjusted_to_tolerances[distances_adjusted_to_tolerances < cmin] = cmin # set overcut

    fig.add_mesh3d(
        x = machining_object.surface_vertices[:, 0],
        y = machining_object.surface_vertices[:, 1],
        z = machining_object.surface_vertices[:, 2],
        colorscale = colorscale,
        cmin=cmin,
        cmax=cmax,
        intensity=distances_adjusted_to_tolerances,

        i = machining_object.surface_triangles[:, 0],
        j = machining_object.surface_triangles[:, 1],
        k = machining_object.surface_triangles[:, 2],
        showscale=True,
        showlegend=True,
        name = 'Machining Error Heatmap'
    )
fig.update_layout(
    showlegend = True,
    scene=dict(
        aspectmode='data'),
        width = 900,
        height = 750
    )

Recall number of paths is also an attribute that you can change without creating a new object

In [ ]:
machining_object.number_of_paths = 10
# show the geometry
fig = go.Figure() # initialize the figure

showheatmap = True

# Add the surface mesh
fig.add_mesh3d(
    x = machining_object.surface_vertices[:, 0],
    y = machining_object.surface_vertices[:, 1],
    z = machining_object.surface_vertices[:, 2],
    i = machining_object.surface_triangles[:, 0],
    j = machining_object.surface_triangles[:, 1],
    k = machining_object.surface_triangles[:, 2],
    showlegend=True,
    name='Surface'
)

# Add the contact curves
X = []
Y = []
Z = []
for env in machining_object.envelopes:
    x,y,z = env.curve(np.linspace(0,1,100))
    X.append(x)
    Y.append(y)
    Z.append(z)
X, Y, Z = np.concatenate(X), np.concatenate(Y), np.concatenate(Z)
fig.add_scatter3d(
    x = X,
    y = Y,
    z = Z,
    mode='markers',
    name='Contact Curves',
    marker=dict(
        size=2,
        color='black',
    ),
    showlegend=True
)

# Add the envelopes
concatenated_envelopes = machining_object.concatenated_envelopes_fun(method='from0to1')
envelopes_vertices = np.asarray(concatenated_envelopes.vertices)
envelopes_triangles = np.asarray(concatenated_envelopes.triangles)
fig.add_mesh3d(
    x = envelopes_vertices[:, 0],
    y = envelopes_vertices[:, 1],
    z = envelopes_vertices[:, 2],
    i = envelopes_triangles[:, 0],
    j = envelopes_triangles[:, 1],
    k = envelopes_triangles[:, 2],
    name='Envelopes',
    opacity=0.5,
    color='red',
    showscale=False,
    showlegend=True
)
# compute the error heatmap and show it only if showheatmap is True
if showheatmap == True:
    errores = real_surface_side_length*machining_object.error_measure_per_vertex()
    colorscale = 'turbo_r'

    cmin, cmax =  real_machining_tolerance*np.array([-1,1,])
    distances_adjusted_to_tolerances = errores.copy()
    distances_adjusted_to_tolerances[distances_adjusted_to_tolerances > cmax] = cmax # set undercut
    distances_adjusted_to_tolerances[distances_adjusted_to_tolerances < cmin] = cmin # set overcut

    fig.add_mesh3d(
        x = machining_object.surface_vertices[:, 0],
        y = machining_object.surface_vertices[:, 1],
        z = machining_object.surface_vertices[:, 2],
        colorscale = colorscale,
        cmin=cmin,
        cmax=cmax,
        intensity=distances_adjusted_to_tolerances,

        i = machining_object.surface_triangles[:, 0],
        j = machining_object.surface_triangles[:, 1],
        k = machining_object.surface_triangles[:, 2],
        showscale=True,
        showlegend=True,
        name = 'Machining Error Heatmap'
    )
fig.update_layout(
    showlegend = True,
    scene=dict(
        aspectmode='data'),
        width = 900,
        height = 750
    )

We can also display the shanks

In [ ]:
n_shanks = 20
shanks = machining_object.discrete_shanks_vectorized(n_shanks = n_shanks)
bottom_shank_points = shanks[:, 0]
top_shank_points = shanks[:, 1]

xs, ys, zs = [], [], []
for A, B in zip(bottom_shank_points, top_shank_points):
    xs.extend([A[0], B[0], None])
    ys.extend([A[1], B[1], None])
    zs.extend([A[2], B[2], None])

trace = go.Scatter3d(
    x=xs,
    y=ys,
    z=zs,
    mode="lines",
    line=dict(
        color="blue",
        width=4
    ),
    name='Shanks Medial Axis',
    showlegend=True
)
fig.add_trace(trace)
fig.show()
